<div style="display:flex; align-items:center"><img src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" width="300"><img src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" width="200"><span style="margin-left:auto; text-align:right"><b>CaixaBank · Advanced Analytics Program</b><br>Python profesional para analítica avanzada</span></div>

# Laboratorio: optimización de un pipeline de datos

Duración: 1,5 horas · Caso: informe mensual de operaciones

## El incidente

El informe diario funciona con el volumen habitual. Al cerrar el mes, el mismo pipeline recibe muchos más registros: el proceso consume demasiada memoria y termina tarde. No cambiaremos todo a la vez. Investigaremos una pregunta cada vez: ¿qué ocupa memoria?, ¿qué datos retenemos sin necesitarlos? y ¿qué efecto tiene cada decisión sobre el tiempo?

Aplicaremos: medir → localizar el coste → cambiar una causa → validar.

## Objetivos

- Auditar memoria total y por columna.
- Diferenciar texto repetido de identificadores casi únicos.
- Compactar tipos sin alterar variables financieras.
- Reducir columnas, filas y copias innecesarias.
- Validar que el informe optimizado conserva la salida.

# Parte guiada — resolver el incidente paso a paso

## 1. Punto de partida: el origen llega completo

El origen contiene los campos del informe y otros que pertenecen solo a la carga diaria. Empezamos con una carga mensual simulada; más adelante comprobaremos si los cambios reducen también el tiempo de cómputo.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pandas.testing import assert_frame_equal

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
def generar_operaciones(n_filas=250_000, semilla=42):
    rng = np.random.default_rng(semilla)
    return pd.DataFrame({
        'operacion_id': np.arange(1, n_filas + 1),
        'region': rng.choice(['norte', 'sur', 'este', 'oeste'], n_filas),
        'canal': rng.choice(['digital', 'oficina', 'partner'], n_filas, p=[.58, .30, .12]),
        'producto': rng.choice(['cuenta', 'tarjeta', 'prestamo', 'inversion'], n_filas),
        'estado': rng.choice(['confirmada', 'pendiente', 'cancelada'], n_filas, p=[.78, .16, .06]),
        'segmento': rng.choice(['retail', 'premium', 'empresa'], n_filas),
        'unidades': rng.integers(1, 120, n_filas),
        'precio_unitario': rng.uniform(12, 2_000, n_filas).round(2),
        'descuento': rng.choice([0.0, .03, .05, .10], n_filas),
        'score_riesgo': rng.uniform(0, 1, n_filas),
        'referencia_externa': [f'EXT-{i:07d}' for i in range(n_filas)],
        'comentario_origen': rng.choice(['carga diaria', 'revision manual', 'origen externo'], n_filas),
    })

operaciones = generar_operaciones()
operaciones.head()

## 2. Primera evidencia: ¿qué explica el consumo?

La medición profunda incluye el coste de objetos Python. La cardinalidad separa el texto repetido de los identificadores casi únicos. Esta auditoría es nuestra primera hipótesis de mejora.

In [ ]:
def memoria_mb(datos):
    return datos.memory_usage(deep=True).sum() / 1024**2

def auditar_memoria(datos):
    tabla = pd.DataFrame({
        'dtype': datos.dtypes.astype(str),
        'valores_unicos': datos.nunique(dropna=False),
        'memoria_mb': datos.memory_usage(deep=True) / 1024**2,
    })
    tabla['ratio_unicos'] = tabla['valores_unicos'] / len(datos)
    return tabla.sort_values('memoria_mb', ascending=False)

def graficar_memoria(tabla, titulo):
    plt.figure(figsize=(9, 5))
    sns.barplot(data=tabla.reset_index(), x='memoria_mb', y='index', color='#0066a6')
    plt.xlabel('Memoria (MB)'); plt.ylabel('Columna'); plt.title(titulo); sns.despine(); plt.show()

auditoria_inicial = auditar_memoria(operaciones)
print(f'Memoria total: {memoria_mb(operaciones):.1f} MB')
auditoria_inicial

In [ ]:
graficar_memoria(auditoria_inicial.head(8), 'Columnas que más memoria consumen')

Region, canal o estado tienen pocos valores repetidos: son candidatas a category. referencia_externa es casi única, por lo que categorizarla no suele ahorrar.

No reducimos precio_unitario a float32: la precisión monetaria es una decisión de negocio, no una microoptimización.

## Caso 1 — El origen ocupa más de lo necesario

**Hipótesis.** Parte de la presión de memoria procede de tipos sobredimensionados y texto repetido. Solo cambiamos tipos seguros: texto repetido a category y contadores a enteros más pequeños. La copia es intencionada: separa el dato de origen del dataset compacto.

In [ ]:
def resumen_reduccion_memoria(antes, despues, etiqueta):
    antes_mb, despues_mb = memoria_mb(antes), memoria_mb(despues)
    return pd.DataFrame([{
        'caso': etiqueta,
        'antes_mb': antes_mb,
        'despues_mb': despues_mb,
        'ahorro_mb': antes_mb - despues_mb,
        'reduccion_pct': 100 * (antes_mb - despues_mb) / antes_mb,
    }])

COLUMNAS_CATEGORIA = ['region', 'canal', 'producto', 'estado', 'segmento', 'comentario_origen']

def compactar_operaciones(datos):
    salida = datos.copy()
    for columna in COLUMNAS_CATEGORIA:
        salida[columna] = salida[columna].astype('category')
    salida['unidades'] = pd.to_numeric(salida['unidades'], downcast='unsigned')
    salida['score_riesgo'] = pd.to_numeric(salida['score_riesgo'], downcast='float')
    return salida

operaciones_compactas = compactar_operaciones(operaciones)
comparacion_tipos = pd.DataFrame({'antes_mb': auditar_memoria(operaciones)['memoria_mb'], 'despues_mb': auditar_memoria(operaciones_compactas)['memoria_mb']})
comparacion_tipos['ahorro_mb'] = comparacion_tipos['antes_mb'] - comparacion_tipos['despues_mb']
resumen_caso_1 = resumen_reduccion_memoria(operaciones, operaciones_compactas, 'tipos del origen')
resumen_caso_1

In [ ]:
for columna in ['region', 'canal', 'producto', 'estado', 'segmento', 'comentario_origen', 'unidades']:
    assert operaciones[columna].astype(str).equals(operaciones_compactas[columna].astype(str))
np.testing.assert_allclose(operaciones['score_riesgo'], operaciones_compactas['score_riesgo'], rtol=1e-6)
print('Tipos compactados y valores validados.')

## Caso 2 — El pipeline todavía retiene demasiados datos

**Punto de partida del caso 2:** usamos exclusivamente operaciones_compactas. Por tanto, el ahorro del caso 1 ya está incorporado y no contaminará esta comparación.

**Hipótesis.** El informe sigue creando intermedios grandes porque copia todas las columnas, calcula para todas las filas y filtra tarde. Ahora solo cambiaremos el orden: seleccionar → filtrar → calcular → agregar.

In [ ]:
COLUMNAS_INFORME = ['region', 'canal', 'estado', 'unidades', 'precio_unitario', 'descuento']

def registrar_etapa(registro, nombre, datos):
    registro.append({'etapa': nombre, 'filas': len(datos), 'columnas': datos.shape[1], 'memoria_mb': memoria_mb(datos)})

def informe_original(datos, devolver_traza=False):
    traza = []
    trabajo = datos.copy(); registrar_etapa(traza, 'copia completa', trabajo)
    trabajo['importe_neto'] = trabajo['unidades'] * trabajo['precio_unitario'] * (1 - trabajo['descuento'])
    registrar_etapa(traza, 'calcular antes de filtrar', trabajo)
    trabajo = trabajo.loc[trabajo['estado'].eq('confirmada')].copy(); registrar_etapa(traza, 'filtrar tarde', trabajo)
    informe = trabajo.groupby(['region', 'canal'], observed=True).agg(operaciones=('operacion_id', 'size'), importe_neto=('importe_neto', 'sum')).reset_index()
    return (informe, pd.DataFrame(traza)) if devolver_traza else informe

def informe_optimizado(datos, devolver_traza=False):
    traza = []
    trabajo = datos.loc[:, COLUMNAS_INFORME]; registrar_etapa(traza, 'seleccionar columnas', trabajo)
    trabajo = trabajo.loc[trabajo['estado'].eq('confirmada')]; registrar_etapa(traza, 'filtrar temprano', trabajo)
    trabajo = trabajo.assign(importe_neto=trabajo['unidades'] * trabajo['precio_unitario'] * (1 - trabajo['descuento']))
    registrar_etapa(traza, 'calcular sobre datos necesarios', trabajo)
    informe = trabajo.groupby(['region', 'canal'], observed=True).agg(operaciones=('estado', 'size'), importe_neto=('importe_neto', 'sum')).reset_index()
    return (informe, pd.DataFrame(traza)) if devolver_traza else informe

In [ ]:
resultado_original, traza_original = informe_original(operaciones_compactas, devolver_traza=True)
resultado_optimizado, traza_optimizada = informe_optimizado(operaciones_compactas, devolver_traza=True)
assert_frame_equal(resultado_original, resultado_optimizado)

pico_original = traza_original['memoria_mb'].max()
pico_optimizado = traza_optimizada['memoria_mb'].max()
resumen_caso_2 = pd.DataFrame([{
    'caso': 'orden del pipeline',
    'antes_mb': pico_original,
    'despues_mb': pico_optimizado,
    'ahorro_mb': pico_original - pico_optimizado,
    'reduccion_pct': 100 * (pico_original - pico_optimizado) / pico_original,
}])
print('El informe optimizado conserva exactamente la misma salida.')
resumen_caso_2

In [ ]:
traza = pd.concat([traza_original.assign(version='original'), traza_optimizada.assign(version='ordenado')], ignore_index=True)
plt.figure(figsize=(10, 5))
sns.barplot(data=traza, x='etapa', y='memoria_mb', hue='version', palette=['#e86e61', '#0066a6'])
plt.xticks(rotation=12, ha='right'); plt.ylabel('Memoria del DataFrame (MB)'); plt.xlabel(''); plt.title('Caso 2: memoria retenida en cada etapa'); sns.despine(); plt.show()
traza

En el caso 2 el indicador es el máximo de memoria retenida por los DataFrames de trabajo. No es el pico total de RAM del proceso, pero permite comparar el efecto del orden del pipeline con la misma entrada compacta.

## 5. ¿Reducir memoria también reduce tiempo?

Ahora, y solo ahora, medimos el caso 2 en tiempo. El objetivo era memoria; el speedup es evidencia adicional, no la razón para mezclar los dos casos.

In [ ]:
def medir(funcion, repeticiones=5):
    funcion()
    tiempos = []
    for _ in range(repeticiones):
        inicio = perf_counter(); funcion(); tiempos.append((perf_counter() - inicio) * 1_000)
    return {'mediana_ms': float(np.median(tiempos)), 'minimo_ms': float(np.min(tiempos))}

medida_original = medir(lambda: informe_original(operaciones_compactas))
medida_optimizada = medir(lambda: informe_optimizado(operaciones_compactas))
speedup_caso_2 = medida_original['mediana_ms'] / medida_optimizada['mediana_ms']
comparacion_tiempo = pd.DataFrame([{'version': 'original', **medida_original}, {'version': 'ordenado', **medida_optimizada}])
plt.figure(figsize=(6, 4))
sns.barplot(data=comparacion_tiempo, x='version', y='mediana_ms', hue='version', palette=['#e86e61', '#0066a6'], legend=False)
plt.ylabel('Mediana (ms)'); plt.xlabel(''); plt.title(f'Caso 2: speedup {speedup_caso_2:.1f}x'); sns.despine(); plt.show()
comparacion_tiempo

# Práctica — resolver un nuevo cierre mensual

Tienes un nuevo origen con el mismo incidente. Los ejercicios siguen las dos partes de la investigación: primero el origen; después el pipeline. En cada cambio calcula el ahorro de memoria y valida antes de continuar.

## Ejercicio 1 — Diagnóstico con evidencia

**Situación.** El nuevo origen tiene 120 000 operaciones y no podemos convertir texto a category de forma indiscriminada.

**Tarea.** Añade a la auditoría una columna candidata_category: vale True cuando el dtype es object y ratio_unicos es menor que 0.05. Muestra únicamente las candidatas, ordenadas por memoria.

**Entrega.** Escribe una frase explicando por qué referencia_externa queda fuera de la política.

In [ ]:
operaciones_practica = generar_operaciones(n_filas=120_000, semilla=7)
auditoria_practica = auditar_memoria(operaciones_practica)
# TODO: crea candidata_category y muestra las columnas candidatas ordenadas por memoria.
# auditoria_practica['candidata_category'] = ...
auditoria_practica

<details><summary><strong>Obtener esqueleto de ayuda para el ejercicio 1</strong></summary>

~~~python
es_texto = auditoria_practica['dtype'].eq('object')
poca_cardinalidad = auditoria_practica['ratio_unicos'].lt(0.05)

# TODO: combina las dos Series booleanas.
auditoria_practica['candidata_category'] = ...

# TODO: filtra y ordena las candidatas por memoria_mb.
candidatas = ...
~~~
</details>

## Ejercicio 2 — Política de compactación

**Situación.** Debes transformar el origen a partir de la evidencia anterior, no a partir de una lista escrita a mano.

**Tarea.** Implementa compactar_practica. Convierte únicamente las candidatas de tu auditoría, reduce unidades y score_riesgo, y conserva precio_unitario como float64. No reutilices COLUMNAS_CATEGORIA.

**Entrega.** La función debe devolver un DataFrame nuevo; no modifiques el argumento de entrada.

In [ ]:
def compactar_practica(datos, auditoria):
    salida = datos.copy()
    # TODO: obtiene candidatas desde auditoria y conviértelas a category.
    # TODO: reduce unidades y score_riesgo; no modifiques precio_unitario.
    return salida

<details><summary><strong>Obtener esqueleto de ayuda para el ejercicio 2</strong></summary>

~~~python
def compactar_practica(datos, auditoria):
    salida = datos.copy()
    candidatas = auditoria.index[auditoria['candidata_category']]

    for columna in candidatas:
        # TODO: convierte esta columna a category.
        salida[columna] = ...

    salida['unidades'] = pd.to_numeric(salida['unidades'], downcast='unsigned')
    # TODO: reduce score_riesgo sin tocar precio_unitario.
    salida['score_riesgo'] = ...
    return salida
~~~
</details>

## Ejercicio 3 — Cuantificar y validar el caso 1

**Situación.** Una reducción de memoria no es válida si cambia los datos que alimentan el informe.

**Tarea.** Aplica tu función y usa resumen_reduccion_memoria para informar ahorro absoluto y porcentual. Valida las columnas de texto convertidas, unidades y score_riesgo; comprueba además que precio_unitario sigue siendo float64.

**Entrega.** Una tabla de reducción y validaciones que se ejecuten sin error.

In [ ]:
# TODO: crea operaciones_practica_compactas, resumen_practica_1 y validaciones.
# operaciones_practica_compactas = ...
# resumen_practica_1 = ...
# assert ...

<details><summary><strong>Obtener esqueleto de ayuda para el ejercicio 3</strong></summary>

~~~python
operaciones_practica_compactas = compactar_practica(
    operaciones_practica, auditoria_practica
)
resumen_practica_1 = resumen_reduccion_memoria(
    operaciones_practica, operaciones_practica_compactas, 'caso 1 práctica'
)

# TODO: valida una columna categórica y unidades.
assert ...
# TODO: comprueba el dtype de precio_unitario y score_riesgo.
~~~
</details>

## Ejercicio 4 — Localizar el máximo del caso 2

**Situación.** El origen ya está compacto, pero el informe sigue generando objetos intermedios grandes.

**Tarea.** Ejecuta informe_original con devolver_traza=True sobre operaciones_practica_compactas. Devuelve una fila llamada cuello_memoria con la etapa de mayor memoria, sus filas, columnas y memoria_mb.

**Entrega.** Indica qué acción del pipeline ocurre en esa etapa y por qué retiene más datos.

In [ ]:
# TODO: genera resultado_practica_original y traza_practica_original.
# TODO: selecciona la fila de mayor memoria_mb con idxmax.
# cuello_memoria = ...

<details><summary><strong>Obtener esqueleto de ayuda para el ejercicio 4</strong></summary>

~~~python
resultado_practica_original, traza_practica_original = informe_original(
    operaciones_practica_compactas, devolver_traza=True
)

indice_pico = traza_practica_original['memoria_mb'].idxmax()
# TODO: selecciona esa fila manteniéndola como DataFrame.
cuello_memoria = ...
cuello_memoria
~~~
</details>

## Ejercicio 5 — Separar una mejora del pipeline

**Situación.** Antes de reescribir el informe completo, aislamos una sola decisión: dejar de arrastrar datos que ya no hacen falta.

**Tarea.** Crea preparar_temprano(datos): debe seleccionar COLUMNAS_INFORME y filtrar confirmadas, en ese orden. Mide la memoria del origen compacto frente al resultado de esta preparación.

**Entrega.** Un resumen_reduccion_memoria que cuantifique el ahorro de esta única intervención.

In [ ]:
def preparar_temprano(datos):
    # TODO: selecciona COLUMNAS_INFORME y filtra estado confirmada.
    return ...

# TODO: mide la reducción de memoria de esta única intervención.

<details><summary><strong>Obtener esqueleto de ayuda para el ejercicio 5</strong></summary>

~~~python
def preparar_temprano(datos):
    trabajo = datos.loc[:, COLUMNAS_INFORME]
    # TODO: conserva solo las operaciones confirmadas.
    trabajo = ...
    return trabajo

preparadas = preparar_temprano(operaciones_practica_compactas)
resumen_preparacion = resumen_reduccion_memoria(
    operaciones_practica_compactas, preparadas, 'selección y filtro temprano'
)
~~~
</details>

## Ejercicio 6 — Completar el caso 2

**Situación.** La preparación temprana funciona; ahora debe integrarse sin perder ninguna regla del informe.

**Tarea.** Implementa informe_practica usando preparar_temprano, calcula importe_neto y agrega por region y canal. Valida que el informe es equivalente a informe_original sobre la misma entrada compacta.

**Entrega.** Una función que respete el contrato del informe y una validación funcional.

In [ ]:
def informe_practica(datos):
    trabajo = preparar_temprano(datos)
    # TODO: crea importe_neto y agrega por region y canal.
    return ...

# TODO: assert_frame_equal(informe_original(...), informe_practica(...))

<details><summary><strong>Obtener esqueleto de ayuda para el ejercicio 6</strong></summary>

~~~python
def informe_practica(datos):
    trabajo = preparar_temprano(datos)
    trabajo = trabajo.assign(
        importe_neto=trabajo['unidades'] * trabajo['precio_unitario'] * (1 - trabajo['descuento'])
    )
    # TODO: agrega por region y canal; cuenta estado y suma importe_neto.
    informe = ...
    return informe

# TODO: valida informe_practica frente a informe_original.
~~~
</details>

## Ejercicio 7 — Defender la decisión

**Situación.** Debes decidir si la modificación merece entrar en el pipeline mensual.

**Criterios de adopción.** Aplicamos la mejora únicamente si: el resultado está validado, el máximo de memoria retenida baja al menos un 20 % y el speedup es como mínimo 1.10x.

**Tarea.** Mide informe_original e informe_practica con la entrada compacta. Calcula speedup y reducción de memoria; después usa los criterios para decidir aplicar, revisar o no aplicar la mejora.

**Entrega.** Un diccionario con métricas, decisión y motivo que permita a otra persona justificar la decisión sin releer el notebook.

In [1]:
CRITERIOS_ADOPCION = {'reduccion_memoria_min_pct': 20, 'speedup_min': 1.10}

# TODO: mide las dos versiones y construye las trazas.
# TODO: calcula speedup_practica y reduccion_memoria_practica.
# TODO: construye la decisión con un if / elif / else.
recomendacion = {
    'reduccion_memoria_pct': ...,
    'speedup_pipeline': ...,
    'resultado_validado': ...,
    'decision': ...,
    'motivo': ...,
}
recomendacion

<details><summary><strong>Obtener esqueleto de ayuda para el ejercicio 7</strong></summary>

~~~python
CRITERIOS_ADOPCION = {'reduccion_memoria_min_pct': 20, 'speedup_min': 1.10}
medida_practica_original = medir(lambda: informe_original(operaciones_practica_compactas))
medida_practica_optimizada = medir(lambda: informe_practica(operaciones_practica_compactas))
speedup_practica = medida_practica_original['mediana_ms'] / medida_practica_optimizada['mediana_ms']

# TODO: genera ambas trazas y calcula reduccion_memoria_practica.
resultado_validado = ...
cumple_memoria = ... >= CRITERIOS_ADOPCION['reduccion_memoria_min_pct']
cumple_tiempo = speedup_practica >= CRITERIOS_ADOPCION['speedup_min']

if resultado_validado and cumple_memoria and cumple_tiempo:
    decision = 'aplicar'
else:
    # TODO: decide si revisar o no aplicar y redacta el motivo.
    decision = ...
~~~
</details>

## Extensión opcional — escalabilidad y presupuesto de memoria

Repite los casos 1 y 2 con 30 000, 120 000 y 480 000 filas. Construye una tabla con tamaño, memoria del origen, memoria compacta y máximo de memoria del pipeline ordenado. ¿Qué tamaño estimarías como límite prudente si tu presupuesto fuera 512 MB? Justifica la estimación, no solo el número.

## Extensión experta — contrato de memoria

Diseña una función verificar_presupuesto(datos, limite_mb) que lance ValueError si el DataFrame supera el límite. Integra la comprobación antes de ejecutar el pipeline y decide qué columnas eliminarías o qué procesamiento por bloques investigarías si falla.

## Preguntas para la discusión

1. ¿Por qué category puede ahorrar memoria en region pero no en una referencia casi única?
2. ¿Por qué mantener precio_unitario en float64 puede ser correcto?
3. ¿Qué diferencia hay entre la memoria de un DataFrame y el pico total de RAM?
4. ¿Por qué el ahorro del caso 1 no demuestra por sí solo que el pipeline está bien ordenado?
5. ¿Qué validación exigirías antes de desplegar esta optimización en un informe regulado?